# 06 - Treinamento DenseNet121 com validação, pré-processamento específico e fine-tuning

Este notebook representa a versão ajustada do treinamento da DenseNet121.

Alterações principais:
- uso dos três subconjuntos: treino, validação e teste;
- `preprocess_input` específico da DenseNet121;
- `data augmentation` aplicado apenas ao treino;
- callbacks monitorando `val_loss`;
- etapa 1: treino da cabeça classificadora;
- etapa 2: fine-tuning das últimas camadas;
- opção para `class_weight` balanceado, suavizado ou desativado.

Este notebook segue a mesma lógica aplicada nas versões ajustadas da ResNet50 e da EfficientNetB0.


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

print("TensorFlow:", tf.__version__)
print("GPUs disponíveis:", tf.config.list_physical_devices("GPU"))

In [ ]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
SPLITS_DIR = DATA_DIR / "splits"

RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
LOGS_DIR = RESULTS_DIR / "logs"
MODELS_DIR = PROJECT_DIR / "models"

for directory in [FIGURES_DIR, METRICS_DIR, LOGS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "DenseNet121"
MODEL_KEY = "densenet121"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 5
AUTOTUNE = tf.data.AUTOTUNE

EPOCHS_HEAD = 10
HEAD_LEARNING_RATE = 1e-3

RUN_FINE_TUNING = True
EPOCHS_FINE_TUNING = 20
FINE_TUNING_LEARNING_RATE = 1e-5
FINE_TUNE_LAST_N_LAYERS = 30

CLASS_WEIGHT_MODE = "balanced"  # opções: "balanced", "sqrt", "none"

class_names = {
    0: "Sem retinopatia",
    1: "Retinopatia leve",
    2: "Retinopatia moderada",
    3: "Retinopatia severa",
    4: "Retinopatia proliferativa"
}

print("Modelo:", MODEL_NAME)
print("Tamanho da imagem:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Modo de class_weight:", CLASS_WEIGHT_MODE)

In [ ]:
train_df = pd.read_csv(SPLITS_DIR / "train_split.csv")
val_df = pd.read_csv(SPLITS_DIR / "val_split.csv")
test_df = pd.read_csv(SPLITS_DIR / "test_split.csv")

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)
print("Teste:", test_df.shape)

display(train_df.head())

In [ ]:
print("Distribuição no treino:")
print(train_df["diagnosis"].value_counts().sort_index())

print("\nDistribuição na validação:")
print(val_df["diagnosis"].value_counts().sort_index())

print("\nDistribuição no teste:")
print(test_df["diagnosis"].value_counts().sort_index())

In [ ]:
missing_train = train_df[~train_df["image_path"].apply(lambda p: Path(str(p)).exists())]
missing_val = val_df[~val_df["image_path"].apply(lambda p: Path(str(p)).exists())]
missing_test = test_df[~test_df["image_path"].apply(lambda p: Path(str(p)).exists())]

print("Imagens ausentes no treino:", len(missing_train))
print("Imagens ausentes na validação:", len(missing_val))
print("Imagens ausentes no teste:", len(missing_test))

if len(missing_train) > 0:
    display(missing_train.head())

if len(missing_val) > 0:
    display(missing_val.head())

if len(missing_test) > 0:
    display(missing_test.head())

if len(missing_train) > 0 or len(missing_val) > 0 or len(missing_test) > 0:
    raise FileNotFoundError("Existem imagens ausentes. Rode novamente: python src/prepare_splits.py")

In [ ]:
classes = np.unique(train_df["diagnosis"].values)

class_weights_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].values
)

class_weights_balanced = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, class_weights_values)
}

class_weights_sqrt = {
    class_id: float(np.sqrt(weight))
    for class_id, weight in class_weights_balanced.items()
}

if CLASS_WEIGHT_MODE == "balanced":
    class_weights = class_weights_balanced
elif CLASS_WEIGHT_MODE == "sqrt":
    class_weights = class_weights_sqrt
elif CLASS_WEIGHT_MODE == "none":
    class_weights = None
else:
    raise ValueError("CLASS_WEIGHT_MODE deve ser 'balanced', 'sqrt' ou 'none'.")

print("Class weights balanceados:")
print(class_weights_balanced)

print("\nClass weights suavizados:")
print(class_weights_sqrt)

print("\nClass weights usados neste experimento:")
print(class_weights)

In [ ]:
def load_image(image_path, label):
    # Não usar str(image_path) aqui.
    # image_path chega como tensor string dentro do pipeline tf.data.
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.one_hot(label, NUM_CLASSES)

    return image, label


def make_dataset(df, shuffle=False):
    # No Windows, converter barras invertidas para barras normais ajuda o TensorFlow.
    image_paths = (
        df["image_path"]
        .astype(str)
        .str.replace("\\", "/", regex=False)
        .values
    )

    labels = df["diagnosis"].astype(int).values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

for images, labels in train_ds.take(1):
    print("Treino - imagens:", images.shape)
    print("Treino - rótulos:", labels.shape)

for images, labels in val_ds.take(1):
    print("Validação - imagens:", images.shape)
    print("Validação - rótulos:", labels.shape)

for images, labels in test_ds.take(1):
    print("Teste - imagens:", images.shape)
    print("Teste - rótulos:", labels.shape)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomZoom(0.05, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomTranslation(0.03, 0.03, fill_mode="constant", fill_value=0.0),
    tf.keras.layers.RandomFlip("horizontal"),
], name="data_augmentation")


def apply_augmentation(images, labels):
    images = data_augmentation(images, training=True)
    return images, labels


train_ds_augmented = train_ds.map(
    apply_augmentation,
    num_parallel_calls=AUTOTUNE
)

for images, labels in train_ds_augmented.take(1):
    print("Treino aumentado - imagens:", images.shape)
    print("Treino aumentado - rótulos:", labels.shape)

In [ ]:
def load_image_for_visualization(image_path):
    image_path = str(image_path)
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image


plt.figure(figsize=(12, 10))
plot_index = 1

for class_id in range(NUM_CLASSES):
    sample_row = train_df[train_df["diagnosis"] == class_id].sample(
        1,
        random_state=SEED
    ).iloc[0]

    image_path = sample_row["image_path"]
    label = int(sample_row["diagnosis"])

    image = load_image_for_visualization(image_path)

    for _ in range(3):
        augmented_image = data_augmentation(
            tf.expand_dims(image, axis=0),
            training=True
        )[0]

        augmented_image_display = tf.clip_by_value(
            augmented_image / 255.0,
            0.0,
            1.0
        )

        plt.subplot(NUM_CLASSES, 3, plot_index)
        plt.imshow(augmented_image_display.numpy())
        plt.title(f"{label} - {class_names[label]}", fontsize=9)
        plt.axis("off")

        plot_index += 1

plt.tight_layout()

augmentation_fig_path = FIGURES_DIR / "data_augmentation_examples_classes.png"
plt.savefig(augmentation_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figura salva em:", augmentation_fig_path)

In [ ]:
base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name="input_image")

x = tf.keras.layers.Lambda(
    preprocess_input,
    name="densenet121_preprocess_input"
)(inputs)

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = tf.keras.layers.Dropout(0.3, name="dropout")(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classification_output"
)(x)

model = tf.keras.Model(inputs, outputs, name="DenseNet121_APTOS")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=HEAD_LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
MODEL_OUTPUT_DIR = MODELS_DIR / MODEL_KEY
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_path = MODEL_OUTPUT_DIR / f"{MODEL_KEY}_best.keras"
final_model_path = MODEL_OUTPUT_DIR / f"{MODEL_KEY}_final.keras"
csv_log_path = LOGS_DIR / f"{MODEL_KEY}_training_log.csv"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        filename=csv_log_path,
        append=False
    )
]

print("Checkpoint:", checkpoint_path)
print("Log CSV:", csv_log_path)

In [ ]:
start_time = time.time()

history_head = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights,
    callbacks=callbacks
)

head_training_time = time.time() - start_time

print(f"Tempo de treinamento da cabeça: {head_training_time:.2f} segundos")

In [ ]:
history_fine = None
fine_tuning_time = 0.0

if RUN_FINE_TUNING:
    base_model.trainable = True

    for layer in base_model.layers[:-FINE_TUNE_LAST_N_LAYERS]:
        layer.trainable = False

    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNING_LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Fine-tuning ativado.")
    print("Últimas camadas liberadas:", FINE_TUNE_LAST_N_LAYERS)
    print("Learning rate:", FINE_TUNING_LEARNING_RATE)

    start_time = time.time()

    history_fine = model.fit(
        train_ds_augmented,
        validation_data=val_ds,
        epochs=EPOCHS_FINE_TUNING,
        class_weight=class_weights,
        callbacks=callbacks
    )

    fine_tuning_time = time.time() - start_time

    print(f"Tempo de fine-tuning: {fine_tuning_time:.2f} segundos")
else:
    print("Fine-tuning desativado.")

In [ ]:
def history_to_dataframe(history, stage_name):
    if history is None:
        return pd.DataFrame()

    df = pd.DataFrame(history.history)
    df["stage"] = stage_name
    df["epoch_in_stage"] = range(1, len(df) + 1)
    return df


history_head_df = history_to_dataframe(history_head, "head")
history_fine_df = history_to_dataframe(history_fine, "fine_tuning")

history_df = pd.concat([history_head_df, history_fine_df], ignore_index=True)
history_df["global_epoch"] = range(1, len(history_df) + 1)

history_path = METRICS_DIR / f"{MODEL_KEY}_history.csv"
history_df.to_csv(history_path, index=False)

display(history_df.tail())

print("Histórico salvo em:", history_path)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["global_epoch"], history_df["loss"], label="Loss treino")

if "val_loss" in history_df.columns:
    plt.plot(history_df["global_epoch"], history_df["val_loss"], label="Loss validação")

plt.title(f"Loss durante o treinamento - {MODEL_NAME}")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_fig_path = FIGURES_DIR / f"{MODEL_KEY}_loss.png"
plt.savefig(loss_fig_path, dpi=300)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(history_df["global_epoch"], history_df["accuracy"], label="Acurácia treino")

if "val_accuracy" in history_df.columns:
    plt.plot(history_df["global_epoch"], history_df["val_accuracy"], label="Acurácia validação")

plt.title(f"Acurácia durante o treinamento - {MODEL_NAME}")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(True)
plt.tight_layout()

acc_fig_path = FIGURES_DIR / f"{MODEL_KEY}_accuracy.png"
plt.savefig(acc_fig_path, dpi=300)
plt.show()

print("Figura de loss salva em:", loss_fig_path)
print("Figura de acurácia salva em:", acc_fig_path)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Loss no teste:", test_loss)
print("Acurácia no teste:", test_accuracy)

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Total de amostras avaliadas:", len(y_true))

In [ ]:
target_names = [class_names[i] for i in range(NUM_CLASSES)]

report = classification_report(
    y_true,
    y_pred,
    target_names=target_names,
    digits=4
)

print(report)

report_path = METRICS_DIR / f"{MODEL_KEY}_classification_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("Relatório salvo em:", report_path)

In [ ]:
cm = confusion_matrix(y_true, y_pred)

cm_path = METRICS_DIR / f"{MODEL_KEY}_confusion_matrix.csv"
pd.DataFrame(cm, index=target_names, columns=target_names).to_csv(cm_path)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title(f"Matriz de Confusão - {MODEL_NAME}")
plt.tight_layout()

cm_fig_path = FIGURES_DIR / f"{MODEL_KEY}_confusion_matrix.png"
plt.savefig(cm_fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Matriz salva em:", cm_path)
print("Figura salva em:", cm_fig_path)

In [ ]:
model.save(final_model_path)

summary_metrics = {
    "model": MODEL_NAME,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "epochs_head_configured": EPOCHS_HEAD,
    "epochs_fine_tuning_configured": EPOCHS_FINE_TUNING if RUN_FINE_TUNING else 0,
    "epochs_completed_total": int(len(history_df)),
    "batch_size": BATCH_SIZE,
    "image_size": list(IMG_SIZE),
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "class_weights": class_weights,
    "split_strategy": "60% treino / 20% validação / 20% teste",
    "preprocess_input": "tensorflow.keras.applications.densenet.preprocess_input",
    "run_fine_tuning": RUN_FINE_TUNING,
    "fine_tune_last_n_layers": FINE_TUNE_LAST_N_LAYERS if RUN_FINE_TUNING else 0,
    "head_learning_rate": HEAD_LEARNING_RATE,
    "fine_tuning_learning_rate": FINE_TUNING_LEARNING_RATE if RUN_FINE_TUNING else None,
    "head_training_time_seconds": float(head_training_time),
    "fine_tuning_time_seconds": float(fine_tuning_time),
    "gpu_available": bool(tf.config.list_physical_devices("GPU")),
    "final_model_path": str(final_model_path),
    "best_checkpoint_path": str(checkpoint_path)
}

summary_path = METRICS_DIR / f"{MODEL_KEY}_summary_metrics.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_metrics, f, indent=4, ensure_ascii=False)

print("Modelo final salvo em:", final_model_path)
print("Resumo salvo em:", summary_path)

summary_metrics

## Observações para comparação

Após executar este notebook, compare os arquivos gerados com os resultados da ResNet50 e da EfficientNetB0:

- `results/metrics/densenet121_classification_report.txt`
- `results/metrics/densenet121_confusion_matrix.csv`
- `results/metrics/densenet121_summary_metrics.json`
- `results/metrics/densenet121_history.csv`
- `results/figures/densenet121_confusion_matrix.png`
- `results/figures/densenet121_loss.png`
- `results/figures/densenet121_accuracy.png`

Métricas prioritárias:
- acurácia no teste;
- F1-score macro;
- F1-score ponderado;
- recall por classe, principalmente classes 1, 2, 3 e 4.
